# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [2]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: c:\Users\Alin\Desktop\ADC\An II\Sem II\AI Engineering\Proiect\echochamber-project-team-5
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [4]:
student_id = "student_4"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [5]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [6]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [11]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15) # completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [12]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [13]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [14]:
# IMPORTANT - SCHIMBA PROMTUL DE MAI JOS PENTRU A SE POTRIVI CU CERINȚELE TALE ȘI ASIGURĂ-TE CĂ RESPECTĂ STRUCTURA SOLICITATĂ
# include în prompt instrucțiuni clare pentru fiecare dintre cele 7 elemente pe care vrei să le extragi și asigură-te că modelul înțelege că trebuie să returneze un JSON valid cu exact acele chei
# inlocueste "..." cu instrucțiuni clare pentru fiecare element
# Prompt de sistem: definește rolul modelului
# poti pune si alte axe de analiza care te intereseaza


SYSTEM_PROMPT = """

Ești un analist specializat în interpretarea comentariilor politice online.
Analizezi comentarii scurte, ironice, emoționale sau ambigue și extragi
informații relevante într-un format structurat.

Trebuie să separi clar:
- poziționarea față de țintă (stance)
de
- tonul discursului (tone).

Răspunsul trebuie să fie exclusiv JSON valid.
Nu adăuga explicații în afara JSON-ului.

"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic și identifică:

1. target:
   Persoana, partidul, instituția, grupul sau actorul politic principal vizat în comentariu.

2. stance:
   Poziționarea autorului față de target.
   Valori posibile:
   - pro
   - anti
   - neutral
   - mixed
   - unclear

3. sentiment:
   Sentimentul general exprimat în comentariu.
   Valori posibile:
   - pozitiv
   - negativ
   - neutru
   - mixt

4. tone:
   Stilul sau tonul discursului.
   Exemple:
   - conspirationist
   - ironic
   - agresiv
   - sarcastic
   - alarmist
   - nationalist
   - batjocoritor
   - furios
   - moralizator
   - neutru

5. topic:
   Tema principală discutată.
   Exemple:
   - alegeri
   - corupție
   - economie
   - justiție
   - politică externă
   - media
   - vaccinuri
   - război
   - imigrație

6. interpretation_problem:
   Posibile dificultăți de interpretare.
   Exemple:
   - ironie
   - sarcasm
   - lipsă context
   - ambiguitate
   - limbaj ofensator
   - referințe indirecte
   - teorii conspiraționiste
   - fără problemă

7. justification:
   O justificare scurtă (1-2 propoziții) pentru etichetele alese.

Important:
- Tonul NU este același lucru cu stance-ul.
  Exemplu:
  un comentariu poate avea ton conspiraționist,
  dar stance anti-guvern.
- Dacă informația nu este clară, folosește "unclear".
- Răspunsul trebuie să fie strict JSON valid.
- Nu adăuga text suplimentar.

... 
Returnează JSON valid cu exact aceste chei:
target, stance, ...
Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [27]:
from openai import OpenAI
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [28]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [29]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,"```json\n{\n ""target"": ""Guvernul României / A..."
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"```json\n{\n ""target"": ""Guvernul României, Ni..."
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","```json\n{\n ""target"": ""George Simion, Klaus ..."
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,"```json\n{\n ""target"": ""George"",\n ""stance"":..."
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...","```json\n{\n ""target"": ""George Simion"",\n ""s..."
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,"```json\n{\n ""target"": ""George (politician)"",..."


# 9. Verificam rezultatele

In [30]:
results_df.model_output[0]

'```json\n{\n  "target": "George Simion",\n  "stance": "pro",\n  "sentiment": "pozitiv",\n  "tone": "batjocoritor",\n  "topic": "politică",\n  "interpretation_problem": "ironie",\n  "justification": "Comentariul folosește un ton exagerat de pozitiv și un set de emoji-uri care par să ironizeze sau să batjocorească o eventuală susținere pentru George Simion ca președinte, sugerând o lipsă de seriozitate în această susținere."\n}\n```'

In [31]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [32]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,George Simion,pro,pozitiv,batjocoritor,politică,ironie,
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,George Simion,pro,pozitiv,aprobator,politică,fără problemă,
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,George Simion,pro,pozitiv,naiv,alegeri,fără problemă,
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,Guvernul României / Autoritățile române,anti,negativ,moralizator,economie / resurse energetice,lipsă context,
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"Guvernul României, Nicușor Dan, Nicolae Ciucă,...",mixed,mixt,"batjocoritor, nationalist, furios",politică internă,"ambiguitate, limbaj ofensator, referințe indir...",
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","George Simion, Klaus Iohannis, Dominic Fritz",mixed,mixt,batjocoritor,politică internă,"ironie, limbaj ofensator",
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,George,pro,pozitiv,apreciativ,politică,fără problemă,
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,George Simion,mixed,mixt,ironic,politică,ironie,
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...",George Simion,pro,pozitiv,entuziast,politică,fără problemă,
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,George (politician),pro,pozitiv,moralizator,politică,fără problemă,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [33]:
csv_path = "rezultate_prompt_comentarii.csv"

results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

csv_path

'rezultate_prompt_comentarii.csv'

In [34]:
check_df = pd.read_csv("rezultate_prompt_comentarii.csv")

check_df.head(10)

,source_channel,video_title,comment_text,model_output
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,"```json\n{\n ""target"": ""Guvernul României / A..."
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"```json\n{\n ""target"": ""Guvernul României, Ni..."
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","```json\n{\n ""target"": ""George Simion, Klaus ..."
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,"```json\n{\n ""target"": ""George"",\n ""stance"":..."
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...","```json\n{\n ""target"": ""George Simion"",\n ""s..."
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,"```json\n{\n ""target"": ""George (politician)"",..."


Promptul separa in general corect sentimentul general de pozitionare fata de targer pentru ca cere explicit doua fielduri diferite: stance si sentiment. Stance descrie relatia autorului fata de subiectul comentariului, pro, anti sau neutri in timp ce sentimentul descrie emotia generala a comentariului, pozitiva, negativa sau neutra. Separarea nu este perfecta in toate cazurile. In comentariile ironice, conspirationiste sau ambigue, modelul poate confunda sentimentul negativ general cu o pozitionare impotriva targetului. Un comentariu poate avea sentiment negativ si ton conspirationist dar targetul sau stanceul pot ramane neclare fara context suplimentar. Promptul face separarea corect, dar rezultatele ar trebui verificate manual, mai ales pentru comentariile sarcastice, ironice sau cu referinte indirecte.